In [1]:
# --- STEP 0: System Setup ---
! git clone https://github.com/sreedharkaranam-ai/edu-101-java-code.git
print("🔧 Installing Java 17, Temporal CLI, and Maven...")
!sudo apt-get update -y > /dev/null
!sudo apt-get install openjdk-17-jdk-headless maven -y > /dev/null
!ls

Cloning into 'edu-101-java-code'...
remote: Enumerating objects: 591, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 591 (delta 79), reused 71 (delta 56), pack-reused 468 (from 2)
Receiving objects: 100% (591/591), 30.25 MiB | 26.34 MiB/s, done.
Resolving deltas: 100% (235/235), done.
🔧 Installing Java 17, Temporal CLI, and Maven...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 35.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
d

In [2]:
# 1. Download using the correct 2026 URL
!curl -sSf https://temporal.download/cli.sh | sh

# 2. Add the binary to the Python environment's PATH
import os
temporal_bin_path = os.path.expanduser("~/.temporalio/bin")
os.environ['PATH'] += f":{temporal_bin_path}"

# 3. Create a system-wide link so all '!temporal' commands work seamlessly
!ln -sf /root/.temporalio/bin/temporal /usr/local/bin/temporal

# 4. Verify it works!
print("\n✅ Verification:")
!temporal --version

%cd /content/edu-101-java-code/example1

temporal: Downloading Temporal CLI latest
temporal: Temporal CLI installed at /root/.temporalio/bin/temporal
temporal: For convenience, we recommend adding it to your PATH
temporal: If using bash, run echo export PATH="\$PATH:/root/.temporalio/bin" >> ~/.bashrc

✅ Verification:
temporal version 1.6.1 (Server 1.30.1, UI 2.45.3)
/content/edu-101-java-code/example1


## --- EXAMPLE 1 ---

In [3]:
import subprocess
import time

# Cleanup
!pkill -9 temporal || true
!pkill -f java || true

print("\n🏗️ Setting up Example 1...")
# Make sure we are in the right folder!
%cd /content/edu-101-java-code/example1

print("🛰️ Terminal 1: Starting Temporal Server (Port 8081)...")
# Popen starts the server in the background and moves to the next line immediately
import subprocess
import time

# 1. Start the Temporal server in the background (equivalent to trailing '&')
print("Starting Temporal server...")
server_process = subprocess.Popen(['temporal', 'server', 'start-dev', '--ui-port', '8081','--db-filename','cluster2.db'])

# 2. Wait 5 seconds for the server to initialize (equivalent to 'sleep 5 &&')
print("Waiting 5 seconds for the server to initialize...")
time.sleep(5)

# 3. Create the search attributes (equivalent to the second command)
print("Registering search attributes...")
create_attrs_cmd = [
    'temporal', 'operator', 'search-attribute', 'create',
    '--namespace', 'default',
    '--type', 'Keyword', '--name', 'CustomKeywordField',
    '--type', 'Int', '--name', 'CustomIntField',
    '--type', 'Double', '--name', 'CustomDoubleField',
    '--type', 'Bool', '--name', 'CustomBoolField',
    '--type', 'Datetime', '--name', 'CustomDatetimeField',
    '--type', 'Text', '--name', 'CustomStringField',
    '--type', 'KeywordList', '--name', 'CustomKeywordListField'
]

# Using subprocess.run will wait for this specific command to finish
try:
    subprocess.run(create_attrs_cmd, check=True)
    print("Search attributes registered successfully!")
except subprocess.CalledProcessError as e:
    print(f"Failed to register search attributes: {e}")

# Note: The server_process is still running in the background here.
# You can now run your Gradle command via subprocess, or keep the script running.

print("👷 Terminal 2: Compiling...")
!mvn clean compile > /dev/null

print("👷 Terminal 2: Starting Java Worker...")
# Run the worker in the background too
worker_process = subprocess.Popen(['mvn', 'exec:java', '-Dexec.mainClass=helloworkflow.SayHelloWorker'])

print("⏳ Waiting 15 seconds for server and worker to start...")
time.sleep(15)

print("🎬 Terminal 3: Running Starter...")
# We use ! for the starter because we DO want the notebook to wait for this one to finish
!mvn exec:java -Dexec.mainClass="helloworkflow.Starter"

^C

🏗️ Setting up Example 1...
/content/edu-101-java-code/example1
🛰️ Terminal 1: Starting Temporal Server (Port 8081)...
Starting Temporal server...
Waiting 5 seconds for the server to initialize...
Registering search attributes...
Search attributes registered successfully!
👷 Terminal 2: Compiling...
👷 Terminal 2: Starting Java Worker...
⏳ Waiting 15 seconds for server and worker to start...
🎬 Terminal 3: Running Starter...
[INFO] Scanning for projects...
[INFO] 
[INFO] ---------------------< com.example:temporal-demo >----------------------
[INFO] Building temporal-demo 1.0-SNAPSHOT
[INFO] --------------------------------[ jar ]---------------------------------
[INFO] 
[INFO] --- exec-maven-plugin:3.6.3:java (default-cli) @ temporal-demo ---
[helloworkflow.Starter.main()] INFO io.temporal.serviceclient.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}}
Workflow result: Hello Tem

## --- EXAMPLE 2 ---


In [4]:
import subprocess
import time

# Cleanup everything first
!pkill -9 temporal || true
!pkill -f java || true

print("\n🏗️ Setting up Example 2...")
# Move to root, clone, then move inside
%cd /content
# Only clone if the folder doesn't exist yet
![ -d "samples-java" ] || git clone https://github.com/temporalio/samples-java
%cd samples-java

print("🛰️ Terminal 4: Starting 2nd Temporal Server (UI 8082)...")
# Popen runs the server in the background without blocking the cell
# Note: I removed '--port 7235' so it defaults to 7233. This allows the sample code to connect!
server_process_2 = subprocess.Popen(['temporal', 'server', 'start-dev', '--ui-port', '8082','--db-filename','cluster3.db'])

# 2. Wait 5 seconds for the server to initialize (equivalent to 'sleep 5 &&')
print("Waiting 5 seconds for the server to initialize...")
time.sleep(5)

# 3. Create the search attributes (equivalent to the second command)
print("Registering search attributes...")
create_attrs_cmd = [
    'temporal', 'operator', 'search-attribute', 'create',
    '--namespace', 'default',
    '--type', 'Keyword', '--name', 'CustomKeywordField',
    '--type', 'Int', '--name', 'CustomIntField',
    '--type', 'Double', '--name', 'CustomDoubleField',
    '--type', 'Bool', '--name', 'CustomBoolField',
    '--type', 'Datetime', '--name', 'CustomDatetimeField',
    '--type', 'Text', '--name', 'CustomStringField',
    '--type', 'KeywordList', '--name', 'CustomKeywordListField'
]

# Using subprocess.run will wait for this specific command to finish
try:
    subprocess.run(create_attrs_cmd, check=True)
    print("Search attributes registered successfully!")
except subprocess.CalledProcessError as e:
    print(f"Failed to register search attributes: {e}")
print("⏳ Waiting 10 seconds for server to boot...")
time.sleep(10)

print("👷 Terminal 5: Building and Executing HelloAccumulator...")
# We use ! here because we WANT the cell to wait for Gradle to finish and print the result
!./gradlew build > /dev/null
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAccumulator

^C

🏗️ Setting up Example 2...
/content
Cloning into 'samples-java'...
remote: Enumerating objects: 7772, done.
remote: Counting objects: 100% (2449/2449), done.
remote: Compressing objects: 100% (776/776), done.
remote: Total 7772 (delta 1920), reused 1735 (delta 1635), pack-reused 5323 (from 1)
Receiving objects: 100% (7772/7772), 3.53 MiB | 12.39 MiB/s, done.
Resolving deltas: 100% (3840/3840), done.
/content/samples-java
🛰️ Terminal 4: Starting 2nd Temporal Server (UI 8082)...
Waiting 5 seconds for the server to initialize...
Registering search attributes...
Search attributes registered successfully!
⏳ Waiting 10 seconds for server to boot...
👷 Terminal 5: Building and Executing HelloAccumulator...
Note: Some input files use or override a deprecated API.
Note: Recompile with -Xlint:deprecation for details.
Note: Some input files use unchecked or unsafe operations.
Note: Recompile with -Xlint:unchecked for details.
/content/samples-java/core/src/test/java/io/temporal/samples/safemes

In [5]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAccumulator

05:31:09.741 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:31:11.480 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAccumulatorTaskQueue", namespace="default", identity=17498@f651fdf9d9b5} 
05:31:11.531 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAccumulatorTaskQueue", namespace="default", identity=17498@f651fdf9d9b5} 
Worker started for task queue: HelloAccumulatorTaskQueue
05:31:12.987 {HelloAccumulatorWorkflow-blue } [signal sendGreeting] INFO  i.t.s.h.HelloAccumulator$AccumulatorWorkflowImpl - received greeting-Greeting [greetingText=Rufus Robot, bucket=blue, greetingKey=key-1] 
05:31:12.987 {HelloAccumulatorWorkflow-red } [signal sendGreeting] INFO  i.t.s.h.HelloAccumulator$AccumulatorWorkflowImpl - receiv

In [6]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloActivity


05:33:23.458 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:33:24.337 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloActivityTaskQueue", namespace="default", identity=18142@f651fdf9d9b5} 
05:33:24.379 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloActivityTaskQueue", namespace="default", identity=18142@f651fdf9d9b5} 
05:33:25.193 {HelloActivityWorkflow e5fa44c2-9a12-3661-9ecf-6df6d529c260} [Activity Executor taskQueue="HelloActivityTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloActivity$GreetingActivitiesImpl - Composing greeting... 
Hello World!


In [7]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloActivityRetry


05:33:30.355 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:33:31.691 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloActivityWithRetriesTaskQueue", namespace="default", identity=18264@f651fdf9d9b5} 
05:33:31.757 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloActivityWithRetriesTaskQueue", namespace="default", identity=18264@f651fdf9d9b5} 
composeGreeting activity is going to fail
05:33:33.328 {HelloActivityWithRetriesWorkflow e77cfe19-6f16-3edc-aef1-be41a11a2574} [Activity Executor taskQueue="HelloActivityWithRetriesTaskQueue", namespace="default": 1] WARN  i.t.i.a.ActivityTaskExecutors$ActivityTaskExecutor - Activity failure. ActivityId=e77cfe19-6f16-3edc-aef1-be41a11a2574, activityType=ComposeGreeting, attempt=1 

In [8]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloActivityExclusiveChoice


05:33:45.286 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:33:46.046 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloActivityChoiceTaskQueue", namespace="default", identity=18418@f651fdf9d9b5} 
05:33:46.083 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloActivityChoiceTaskQueue", namespace="default", identity=18418@f651fdf9d9b5} 
Order results: Ordered 8 Apples...Ordered 1 Cherries...Ordered 5 Bananas...Ordered 4 Oranges...


In [9]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAsync


05:33:56.276 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:33:57.143 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAsyncActivityTaskQueue", namespace="default", identity=18548@f651fdf9d9b5} 
05:33:57.191 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAsyncActivityTaskQueue", namespace="default", identity=18548@f651fdf9d9b5} 
Hello World!
Bye World!


In [10]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloParallelActivity


05:34:03.576 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:34:04.396 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloParallelActivityTaskQueue", namespace="default", identity=18680@f651fdf9d9b5} 
05:34:04.440 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloParallelActivityTaskQueue", namespace="default", identity=18680@f651fdf9d9b5} 
Hello John!
Hello Mary!
Hello Michael!
Hello Jennet!


In [11]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAsyncActivityCompletion


05:34:14.297 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:34:15.141 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAsyncActivityCompletionTaskQueue", namespace="default", identity=18813@f651fdf9d9b5} 
05:34:15.194 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAsyncActivityCompletionTaskQueue", namespace="default", identity=18813@f651fdf9d9b5} 
Hello World!


In [12]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAsyncLambda


05:34:20.895 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:34:21.614 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAsyncLambdaTaskQueue", namespace="default", identity=18942@f651fdf9d9b5} 
05:34:21.651 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAsyncLambdaTaskQueue", namespace="default", identity=18942@f651fdf9d9b5} 
Hello World!
Hello World!


In [13]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloDetachedCancellationScope


05:34:30.587 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:34:31.914 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloDetachedCancellationTaskQueue", namespace="default", identity=19072@f651fdf9d9b5} 
05:34:31.957 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloDetachedCancellationTaskQueue", namespace="default", identity=19072@f651fdf9d9b5} 
05:34:34.854 {HelloDetachedCancellationWorkflow fbb7bd29-4ae8-38a0-a754-2eb5bb6ae730} [Activity Executor taskQueue="HelloDetachedCancellationTaskQueue", namespace="default": 1] INFO  i.t.i.a.ActivityTaskExecutors$ActivityTaskExecutor - Activity canceled. ActivityId=fbb7bd29-4ae8-38a0-a754-2eb5bb6ae730, activityType=SayHello, attempt=1 
Goodbye John!


In [14]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloChild


05:34:39.874 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:34:40.707 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloChildTaskQueue", namespace="default", identity=19212@f651fdf9d9b5} 
Hello World!


In [15]:
!timeout 240s ./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloCron
# taking too long

05:34:50.475 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:34:51.737 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloCronTaskQueue", namespace="default", identity=19338@f651fdf9d9b5} 
05:34:51.778 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloCronTaskQueue", namespace="default", identity=19338@f651fdf9d9b5} 
Started workflow_id: "HelloCronWorkflow"
run_id: "019cd63d-ac29-766f-b31f-466fbd253787"

From HelloCronWorkflow: Hello World!
From HelloCronWorkflow: Hello World!
From HelloCronWorkflow: Hello World!


In [16]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloDynamic


05:38:48.677 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:38:50.016 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloDynamicTaskQueue", namespace="default", identity=20390@f651fdf9d9b5} 
05:38:50.126 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloDynamicTaskQueue", namespace="default", identity=20390@f651fdf9d9b5} 
DynamicACT: Hello John from: DynamicWF


In [17]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloEagerWorkflowStart


05:39:03.301 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:39:04.284 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloEagerWorkflowStartTaskQueue", namespace="default", identity=20545@f651fdf9d9b5} 
05:39:04.334 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloEagerWorkflowStartTaskQueue", namespace="default", identity=20545@f651fdf9d9b5} 
05:39:05.328 {HelloEagerWorkflowStartWorkflow 87f904e5-5321-3627-82c1-197020888085} [Activity Executor taskQueue="HelloEagerWorkflowStartTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloEagerWorkflowStart$GreetingLocalActivitiesImpl - Composing greeting... 
Hello World!


In [18]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloException


05:39:10.677 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:39:11.608 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloExceptionTaskQueue", namespace="default", identity=20672@f651fdf9d9b5} 
05:39:11.667 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloExceptionTaskQueue", namespace="default", identity=20672@f651fdf9d9b5} 
05:39:13.346 {2e5f792e-0a26-3925-b7ed-ecd25b2efd06 b863af4d-acce-3e41-aa3e-890bcd292553} [Activity Executor taskQueue="HelloExceptionTaskQueue", namespace="default": 1] WARN  i.t.i.a.ActivityTaskExecutors$ActivityTaskExecutor - Activity failure. ActivityId=b863af4d-acce-3e41-aa3e-890bcd292553, activityType=ComposeGreeting, attempt=1 
java.io.IOException: Hello World!
	at io.temporal.samples.hello.Hel

In [19]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloLocalActivity


05:39:21.431 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:39:22.331 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloLocalActivity", namespace="default", identity=20813@f651fdf9d9b5} 
05:39:22.388 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloLocalActivity", namespace="default", identity=20813@f651fdf9d9b5} 
Hello World!


In [20]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloPeriodic


05:39:28.676 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:39:29.508 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloPeriodicTaskQueue", namespace="default", identity=20936@f651fdf9d9b5} 
05:39:29.547 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloPeriodicTaskQueue", namespace="default", identity=20936@f651fdf9d9b5} 
Started a new GreetingWorkflow.
Observing the workflow execution for 20 seconds.
From HelloPeriodicWorkflow: Hello World! I will sleep for 5573 milliseconds and then I will greet you again.
From HelloPeriodicWorkflow: Hello World! I will sleep for 5007 milliseconds and then I will greet you again.
From HelloPeriodicWorkflow: Hello World! I will sleep for 5183 milliseconds and then I will greet you agai

In [21]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloPolymorphicActivity


05:39:55.346 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:39:56.161 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloPolymorphicActivityTaskQueue", namespace="default", identity=21138@f651fdf9d9b5} 
05:39:56.213 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloPolymorphicActivityTaskQueue", namespace="default", identity=21138@f651fdf9d9b5} 
Hello World!
Bye World!



In [22]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloQuery


05:40:02.572 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:40:04.238 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloQueryTaskQueue", namespace="default", identity=21256@f651fdf9d9b5} 
Hello World!
Bye World!


In [23]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSaga


05:40:12.893 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:40:13.753 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSagaTaskQueue", namespace="default", identity=21387@f651fdf9d9b5} 
05:40:13.790 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSagaTaskQueue", namespace="default", identity=21387@f651fdf9d9b5} 
ActivityOperationImpl.execute() is called with amount 10
ActivityOperationImpl.execute() is called with amount 20
Other compensation logic in main workflow.
ActivityCompensationImpl.compensate() is called with amount -20
ActivityCompensationImpl.compensate() is called with amount -10


In [24]:
!timeout 300s ./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSchedules

05:40:22.669 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:40:24.404 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloScheduleTaskQueue", namespace="default", identity=21516@f651fdf9d9b5} 
05:40:24.503 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloScheduleTaskQueue", namespace="default", identity=21516@f651fdf9d9b5} 
From HelloScheduleWorkflow-2026-03-10T05:40:24Z: Hello World from HelloSchedule scheduled at 2026-03-10T05:40:24Z!
From HelloScheduleWorkflow-2026-03-10T05:40:30Z: Hello World from HelloSchedule scheduled at 2026-03-10T05:40:30Z!
From HelloScheduleWorkflow-2026-03-10T05:40:35Z: Hello World from HelloSchedule scheduled at 2026-03-10T05:40:35Z!
From HelloScheduleWorkflow-2026-03-10T05:40:40Z: Hello World

In [25]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSignal


05:41:20.641 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:41:21.405 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSignalTaskQueue", namespace="default", identity=21851@f651fdf9d9b5} 
[Hello World!, Hello Universe!]


In [26]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSearchAttributes

05:41:30.238 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:41:31.517 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSearchAttributesTaskQueue", namespace="default", identity=21966@f651fdf9d9b5} 
05:41:31.549 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSearchAttributesTaskQueue", namespace="default", identity=21966@f651fdf9d9b5} 
In workflow we get CustomKeywordField is: keys
Hello SearchAttributes!


In [27]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloTypedSearchAttributes


05:41:37.413 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:41:38.192 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloTypedSearchAttributesTaskQueue", namespace="default", identity=22096@f651fdf9d9b5} 
05:41:38.226 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloTypedSearchAttributesTaskQueue", namespace="default", identity=22096@f651fdf9d9b5} 
Hello TypedSearchAttributes how are you doing?


In [28]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSideEffect



05:41:45.712 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:41:47.593 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSideEffectTaskQueue", namespace="default", identity=22219@f651fdf9d9b5} 
05:41:47.678 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSideEffectTaskQueue", namespace="default", identity=22219@f651fdf9d9b5} 
Hello World
Hello World


In [29]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloUpdate


05:41:54.332 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:41:55.232 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloUpdateTaskQueue", namespace="default", identity=22354@f651fdf9d9b5} 
05:41:55.278 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloUpdateTaskQueue", namespace="default", identity=22354@f651fdf9d9b5} 
05:41:56.154 {HelloUpdateWorkflow a755440f-8831-399e-a36b-a22ce4b1f2fc} [Activity Executor taskQueue="HelloUpdateTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloActivity$GreetingActivitiesImpl - Composing greeting... 
05:41:56.442 {HelloUpdateWorkflow 866f0faf-edbc-30c8-91a3-e9f0f4eebf21} [Activity Executor taskQueue="HelloUpdateTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloActivity$Greetin

In [30]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSignalWithTimer


05:42:05.915 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:42:07.432 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSignalWithTimerTaskQueue", namespace="default", identity=22485@f651fdf9d9b5} 
05:42:07.481 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSignalWithTimerTaskQueue", namespace="default", identity=22485@f651fdf9d9b5} 
Processing value: Value 2
Processing value: Value 5
Processing value: Value 8
Processing value: Value 10
05:42:33.348 {HelloSignalWithTimerWorkflow } [workflow-method-HelloSignalWithTimerWorkflow-f142f25b-dfd2-429e-85e5-1b44f2f0ed98] INFO  i.t.s.h.HelloSignalWithTimer$SignalWithTimerWorkflowImpl - Timer canceled via exit signal 
Processing value: Value 11


In [31]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSignalWithStartAndWorkflowInit

05:42:41.709 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:42:42.817 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloWithInitTaskQueue", namespace="default", identity=22723@f651fdf9d9b5} 
05:42:42.862 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloWithInitTaskQueue", namespace="default", identity=22723@f651fdf9d9b5} 
Result: Hello Michael Jordan,Hello John Stockton
05:42:44.021 {without-init } [signal addGreeting] WARN  i.t.i.sync.WorkflowExecutionHandler - Workflow execution failure WorkflowId='without-init', RunId=995a3130-76fc-42e2-b41a-f91bea4582a7, WorkflowType='MyWorkflowNoInit' 
java.lang.NullPointerException: Cannot invoke "java.util.List.add(Object)" because "this.peopleToGreet" is null
	at io.temporal.sam

In [32]:
# Cleanup
!pkill -9 temporal || true
!pkill -f java || true

^C
